# Previsão de gastos - Regressão

Ideia: usar os gastos mensais + ibovespa pra tentar prever o gasto do próximo mês.
A hipótese é que em meses de bolsa em alta eu gasto mais (confiança do consumidor, etc).

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

DB_PATH = "data/processed/financeiro.db"

In [ ]:
conn = sqlite3.connect(DB_PATH)

# puxar só os gastos direto no SQL
gastos = pd.read_sql("""
    SELECT data, valor, categoria, tipo
    FROM transacoes
    WHERE LOWER(tipo) IN ('despesa', 'débito', 'debito')
      AND anomalia = 0
""", conn, parse_dates=["data"])

print(f"gastos: {len(gastos)}")
gastos.head()

## Agregação mensal

In [ ]:
# total de gastos por mês
gastos["ano_mes"] = gastos["data"].dt.strftime("%Y-%m")

mensal = gastos.groupby("ano_mes").agg(
    total_gasto=("valor", "sum"),
    qtd_transacoes=("valor", "count"),
    media_transacao=("valor", "mean"),
    max_transacao=("valor", "max"),
).reset_index()

mensal["data"] = pd.to_datetime(mensal["ano_mes"] + "-01")

print(f"{len(mensal)} meses")
mensal.head()

In [ ]:
# ibovespa mensal via SQL
ibov_mensal = pd.read_sql("""
    SELECT strftime('%Y-%m', data) as ano_mes,
           AVG("Close") as ibov_media,
           -- não dá pra calcular std no sqlite direto, vou fazer no pandas
           COUNT(*) as pregoes
    FROM ibovespa
    WHERE "Close" IS NOT NULL
    GROUP BY strftime('%Y-%m', data)
    ORDER BY ano_mes
""", conn)

# volatilidade precisa do pandas mesmo
df_ibov = pd.read_sql("SELECT data, retorno_diario FROM ibovespa", conn, parse_dates=["data"])
df_ibov["ano_mes"] = df_ibov["data"].dt.strftime("%Y-%m")
vol = df_ibov.groupby("ano_mes")["retorno_diario"].std().reset_index()
vol.columns = ["ano_mes", "ibov_vol"]

ibov_mensal = ibov_mensal.merge(vol, on="ano_mes")
ibov_mensal.head()

In [ ]:
# juntar tudo
dados = mensal.merge(ibov_mensal, on="ano_mes", how="inner")

# orçamento via SQL
orc = pd.read_sql("""
    SELECT strftime('%Y-%m', "Mês") as ano_mes,
           Limite as orcamento
    FROM orcamento
""", conn)

dados = dados.merge(orc, on="ano_mes", how="left")

conn.close()

print(f"dataset final: {len(dados)} meses")
dados.head()

## Features

- `total_gasto_lag1`: gasto do mês anterior
- `ibov_media`: média de fechamento do Ibovespa no mês
- `ibov_vol`: volatilidade (std retorno diário)
- `orcamento`: limite de orçamento definido
- `mes`: sazonalidade (dezembro gasta mais, janeiro menos...)

In [ ]:
dados["total_gasto_lag1"] = dados["total_gasto"].shift(1)
dados["total_gasto_lag2"] = dados["total_gasto"].shift(2)
dados["mes"] = dados["data"].dt.month

# dropar as 2 primeiras linhas (sem lag)
dados = dados.dropna().reset_index(drop=True)

features = ["total_gasto_lag1", "total_gasto_lag2", "ibov_media", "ibov_vol", "mes"]

# adicionar orcamento se tiver
if dados["orcamento"].notna().sum() > 5:
    dados["orcamento"] = dados["orcamento"].fillna(dados["orcamento"].median())
    features.append("orcamento")
    print("usando orcamento como feature")
else:
    print("orcamento tem pouco dado, ignorando")

X = dados[features]
y = dados["total_gasto"]

print(f"features: {features}")
print(f"amostras: {len(X)}")

## Regressão Linear (baseline)

In [ ]:
# vou testar com 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)  # shuffle=False pq é série temporal

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

r2_lr = r2_score(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print(f"Linear Regression")
print(f"  R²:   {r2_lr:.4f}")
print(f"  RMSE: R$ {rmse_lr:,.2f}")
print(f"  MAE:  R$ {mae_lr:,.2f}")

In [ ]:
# coeficientes para entender o que tá pesando
coefs = pd.Series(lr.coef_, index=features).sort_values(key=abs, ascending=False)
print("Coeficientes:")
print(coefs)

## Random Forest

Vou testar um RF pra ver se melhora. Com poucos dados assim pode overfitar,
mas pelo menos serve de comparação.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(f"Random Forest")
print(f"  R²:   {r2_rf:.4f}")
print(f"  RMSE: R$ {rmse_rf:,.2f}")
print(f"  MAE:  R$ {mae_rf:,.2f}")

# importância
imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("\nImportância:")
print(imp)

In [ ]:
# comparação visual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, nome, r2 in [
    (axes[0], y_pred_lr, "Linear Regression", r2_lr),
    (axes[1], y_pred_rf, "Random Forest", r2_rf),
]:
    ax.plot(y_test.values, "o-", label="Real", markersize=6)
    ax.plot(y_pred, "s--", label="Previsto", markersize=6, alpha=0.8)
    ax.set_title(f"{nome} (R²={r2:.3f})")
    ax.set_xlabel("Mês (teste)")
    ax.set_ylabel("Total gastos (R$)")
    ax.legend()

plt.tight_layout()
plt.savefig("outputs/regressao_comparacao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# feature importance do RF
fig, ax = plt.subplots(figsize=(8, 4))
imp.plot.barh(ax=ax, color="steelblue")
ax.set_title("Feature Importance - Random Forest")
ax.set_xlabel("Importância")
plt.tight_layout()
plt.savefig("outputs/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## Resumo

| Modelo | R² | RMSE | MAE |
|--------|-----|------|-----|

O lag de 1 mês é de longe a feature mais importante (faz sentido,
gastos são bem autocorrelacionados).

O Ibovespa tem relevância baixa — pode ser que com mais dados apareça algo,
mas com só ~24 meses é difícil confirmar a hipótese.

In [ ]:
# salvar resultados
resultados = pd.DataFrame({
    "modelo": ["Linear Regression", "Random Forest"],
    "r2": [r2_lr, r2_rf],
    "rmse": [rmse_lr, rmse_rf],
    "mae": [mae_lr, mae_rf],
})
resultados.to_csv("outputs/resultados_regressao.csv", index=False)
print(resultados.to_string(index=False))